# Henry Hub: curva forward y valuación de un swap

**Daniela Ibarra Gatica**

Análisis de la estacionalidad de la curva forward del gas natural en Henry Hub y valuación
de un swap fijo contra flotante a once meses.

Este notebook extiende a un commodity físico un marco de valuación que construí antes para
swaps de tasa de interés sobre TIIE Fondeo. La mecánica se traslada casi igual —curva de
descuento, factor de anualidad, tasa par—, pero la forma de la curva forward no, y ahí está
el punto del ejercicio.

**Contenido**
1. Preparación
2. Historia del precio spot (1997–2026)
3. Curva forward y estacionalidad
4. Valuación del swap
5. Sensibilidad a la tasa de descuento


## 1. Preparación


In [ ]:
%pip install pandas matplotlib --quiet


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Carpetas de salida
Path("output").mkdir(exist_ok=True)
Path("datos").mkdir(exist_ok=True)


## 2. Historia del precio spot

Precio spot diario de Henry Hub desde enero de 1997, publicado por la Administración de
Información Energética de Estados Unidos (EIA) y distribuido a través de FRED bajo la serie
`DHHNGSP`, en dólares por millón de BTU (MMBtu).

Sobre la procedencia: la EIA publica este precio a través de más de un producto, alimentados
por reporteros de precios distintos, y no coinciden en días individuales. Aquí uso la serie
distribuida por FRED.


In [ ]:
URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DHHNGSP"

hh = pd.read_csv(URL)
hh.columns = ["fecha", "precio"]
hh["fecha"]  = pd.to_datetime(hh.fecha)
hh["precio"] = pd.to_numeric(hh.precio, errors="coerce")   # FRED marca los días sin dato con '.'
hh = hh.dropna().reset_index(drop=True)

print(f"Observaciones: {len(hh)}")
print(f"Rango: {hh.fecha.min().date()} a {hh.fecha.max().date()}")
hh.tail()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hh.fecha, hh.precio, linewidth=0.5, color="#1F3864")
ax.set_title("Henry Hub natural gas spot price", loc="left", fontsize=13)
ax.set_ylabel("USD/MMBtu")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.02, "Source: EIA via FRED (DHHNGSP)", fontsize=8, color="gray")
fig.tight_layout()
fig.savefig("output/01_spot_historico.png", dpi=150, bbox_inches="tight")


### Años extremos


In [ ]:
hh["anio"] = hh.fecha.dt.year
hh.groupby("anio").precio.max().sort_values(ascending=False).head(6)


Tres episodios dominan la serie, y sus causas son estructuralmente distintas:

- **2020 — colapso de demanda.** Los confinamientos por COVID-19 hundieron el consumo
  industrial y comercial.
- **Febrero de 2021 — choque de oferta local.** La tormenta invernal Uri congeló cabezas de
  pozo y líneas de recolección en Texas, recortando la producción justo cuando se disparaba
  la demanda para calefacción y generación eléctrica.
- **2022 — choque de demanda global.** Tras la invasión de Ucrania, Europa sustituyó el gas
  ruso por ducto con gas natural licuado transportado por mar. Como Estados Unidos es
  exportador de GNL, la demanda europea se transmitió al precio interno: evidencia de que
  el gas dejó de ser un mercado regional para volverse uno globalmente conectado.

Uno fue una caída de demanda, otro una restricción física de oferta, otro un desplazamiento
geopolítico de demanda. En commodities el precio responde a restricciones de producción,
almacenamiento y transporte, no solo a expectativas.


## 3. Curva forward

Precios de los once contratos consecutivos del NYMEX para Henry Hub (raíz `NG`), de octubre
de 2026 a agosto de 2027, capturados de TradingView al cierre del 28 de agosto de 2026,
14:59 GMT-6.

Se excluye el contrato de septiembre de 2026: a días de su vencimiento, su precio está
distorsionado por efectos de roll.

Son últimos precios operados, **no** settlements oficiales del CME. Los settlements los
calcula la bolsa sobre la ventana de cierre y son el insumo correcto para valuación en
producción y para el cálculo de márgenes. Para un ejercicio de forma de curva la diferencia
es inmaterial, pero no es cero.


In [ ]:
csv = """contrato,mes_entrega,precio
NGV2026,2026-10-01,2.888
NGX2026,2026-11-01,3.035
NGZ2026,2026-12-01,3.535
NGF2027,2027-01-01,3.937
NGG2027,2027-02-01,3.593
NGH2027,2027-03-01,2.921
NGJ2027,2027-04-01,2.786
NGK2027,2027-05-01,2.800
NGM2027,2027-06-01,2.942
NGN2027,2027-07-01,3.150
NGQ2027,2027-08-01,3.220
"""

with open("datos/curva_hh.csv", "w") as f:
    f.write(csv)

curva = pd.read_csv("datos/curva_hh.csv", parse_dates=["mes_entrega"])
curva = curva.sort_values("mes_entrega").reset_index(drop=True)
curva


### Estacionalidad


In [ ]:
pico  = curva.loc[curva.precio.idxmax()]
piso  = curva.loc[curva.precio.idxmin()]
strip = curva.precio.mean()

print(f"Máximo:  {pico.contrato} ({pico.mes_entrega:%b %Y})  {pico.precio:.3f}")
print(f"Mínimo:  {piso.contrato} ({piso.mes_entrega:%b %Y})  {piso.precio:.3f}")
print(f"Amplitud: {pico.precio - piso.precio:.3f} USD/MMBtu "
      f"({(pico.precio/piso.precio - 1)*100:.1f}% sobre el mínimo)")
print(f"Strip:   {strip:.4f} USD/MMBtu")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(curva.mes_entrega, curva.precio, marker="o", color="#1F3864", linewidth=1.6)
ax.axhline(strip, color="gray", linestyle="--", linewidth=0.8)
ax.annotate(f"Strip avg {strip:.3f}", (curva.mes_entrega.iloc[0], strip),
            textcoords="offset points", xytext=(5, 6), fontsize=8, color="gray")

for _, r in curva.iterrows():
    ax.annotate(f"{r.precio:.2f}", (r.mes_entrega, r.precio),
                textcoords="offset points", xytext=(0, 11),
                ha="center", fontsize=7.5, color="#444444")

ax.set_title("Henry Hub natural gas forward curve", loc="left", fontsize=13)
ax.set_ylabel("USD/MMBtu")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.02,
         "Source: TradingView, NYMEX Henry Hub Natural Gas Futures (NG), "
         "close of 28 Aug 2026 14:59 GMT-6",
         fontsize=7.5, color="gray")
fig.tight_layout()
fig.savefig("output/02_curva_forward.png", dpi=150, bbox_inches="tight")


La curva no dibuja una apuesta direccional, dibuja el año de consumo: sube hacia el pico de
calefacción de enero, cae con fuerza durante marzo, toca fondo en abril —cuando terminó el
invierno y todavía no empieza la demanda de enfriamiento— y se recupera hacia el verano por
consumo de aire acondicionado.

El movimiento mensual más pronunciado es de febrero a marzo, **67 centavos**: el mercado
poniéndole precio al fin de la temporada de calefacción.


## 4. Valuación del swap

Un swap fijo contra flotante sobre Henry Hub a once meses, con liquidación mensual y 10,000
MMBtu por mes. Un comprador de gas que paga fijo y recibe flotante queda cubierto ante un
alza de precio.

El problema es encontrar el precio fijo `K` que hace que el swap valga cero al inicio:

$$VP_{flotante} = \sum_i F_i \cdot V \cdot df_i \qquad VP_{fijo} = K \cdot V \cdot \sum_i df_i$$
$$K = \frac{VP_{flotante}}{V \cdot \sum_i df_i}$$

**Supuestos.** Tasa libre de riesgo en dólares plana de 4% (una valuación en producción
usaría una curva de descuento SOFR). Liquidación a fin del mes de entrega.


In [ ]:
VALUACION = pd.Timestamp("2026-08-28")   # fecha de la curva
TASA      = 0.04                          # supuesto: tasa libre de riesgo USD, plana
VOLUMEN   = 10_000                        # MMBtu por mes

curva["fecha_pago"] = curva.mes_entrega + pd.offsets.MonthEnd(0)
curva["t"]  = (curva.fecha_pago - VALUACION).dt.days / 365
curva["df"] = 1 / (1 + TASA) ** curva.t

print(f"Factores de descuento: de {curva.df.min():.4f} a {curva.df.max():.4f}")


### Pata flotante


In [ ]:
curva["flujo_flotante"] = curva.precio * VOLUMEN
curva["vp_flotante"]    = curva.flujo_flotante * curva.df

vp_flotante = curva.vp_flotante.sum()
print(f"VP de la pata flotante: {vp_flotante:,.2f} USD")


### Precio fijo de equilibrio

`anualidad` es la anualidad nocional descontada: el mismo objeto que aparece como factor de
anualidad en un swap de tasa de interés.


In [ ]:
anualidad   = (VOLUMEN * curva.df).sum()   # anualidad nocional descontada
precio_fijo = vp_flotante / anualidad

print(f"Precio fijo de equilibrio: {precio_fijo:.4f} USD/MMBtu")
print(f"Strip promedio simple:     {curva.precio.mean():.4f} USD/MMBtu")
print(f"Diferencia:                {(precio_fijo - curva.precio.mean())*100:+.2f} centavos")


**Por qué difieren.** El strip promedio le da el mismo peso a cada mes; el swap descuenta
cada flujo según su propio plazo. El precio fijo de equilibrio es entonces un promedio de la
curva forward *ponderado por factores de descuento*, no aritmético.

Como en esta curva los meses caros caen al principio y los baratos al final, el descuento
castiga más a los baratos y jala el precio fijo ligeramente por encima del strip.


### Comprobación

Si `K` está bien calculado, el swap vale cero al inicio.


In [ ]:
curva["vp_fijo"] = precio_fijo * VOLUMEN * curva.df
mtm = (curva.vp_flotante - curva.vp_fijo).sum()
print(f"MtM al inicio: {mtm:.6f}   (se espera ~0)")

curva[["contrato", "precio", "t", "df", "vp_flotante", "vp_fijo"]].round(4)


## 5. Sensibilidad a la tasa de descuento

La brecha entre el precio de equilibrio y el strip simple es pequeña solo porque el plazo es
corto. Al subir la tasa, se abre.


In [ ]:
for tasa in [0.00, 0.04, 0.10, 0.20]:
    df  = 1 / (1 + tasa) ** curva.t
    k   = (curva.precio * VOLUMEN * df).sum() / (VOLUMEN * df).sum()
    dif = (k - curva.precio.mean()) * 100
    print(f"tasa {tasa:5.0%}   precio fijo {k:.4f}   vs strip {curva.precio.mean():.4f}   "
          f"dif {dif:+.2f} centavos")


A tasa de descuento cero el precio fijo cae exactamente sobre el strip promedio, que es la
comprobación aritmética de que el mecanismo es el descuento y nada más.

---

## Limitaciones y siguientes pasos

- Tasa de descuento plana en lugar de una curva SOFR.
- Valuación determinista. Es lo correcto para un swap plain vanilla: la curva forward ya
  incorpora la expectativa y la prima de riesgo del mercado. La simulación estocástica solo
  sería necesaria con opcionalidad o para perfiles de exposición de contraparte.
- **Extensión natural:** agregar CVA/DVA a este swap, reutilizando el bootstrapping de
  curvas CDS y la simulación de exposición con Hull-White de mi proyecto de swaps de tasa
  de interés, que es el punto donde la valuación determinista deja de ser suficiente.
